# A3 · Dates and New Variables

*Path A: Write It Yourself, notebook 3 of 6 · about 20 to 30 minutes*

**In this notebook you will learn to:**

- Compare and subtract dates
- Work out who was exposed to a drug before their diagnosis
- Test your logic on a few cases where you know the answer
- Create the variables needed for analysis, and save your data

**Before this notebook:** [A2: One Row per Person](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/A2_One_Row_per_Person.ipynb).

> **Before you start:** click **File > Save a copy in Drive** and work in your copy. Run a code cell by clicking the play button to its left, or press **Shift + Enter**. Go in order from top to bottom.

## 1. Where we are

You have one table, `combined_df`, with one row per person. It contains each person's first diagnosis date and first prescription date, where they exist. Now you need to turn those dates into the variables the analysis needs.

Run the setup cell. The lines printed by `library(tidyverse)` and `read_csv()` are normal. It loads a saved copy of the table you built in notebook A2.

In [ ]:
library(tidyverse)

# Catch-up: the merged table from notebook A2
# (a minus sign inside select() removes a column instead of keeping it)
combined_df <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/final_dataset.csv") %>%
  select(-true_exposure_status, -SBP)

head(combined_df)

## 2. Arithmetic with dates

The two date columns are marked `<date>`, so R knows they are dates and not text. That means you can subtract them. Here are the people who have both dates, with the number of days from first prescription to first diagnosis.

In [ ]:
combined_df %>%
  filter(!is.na(drug_exposure_start_date) & !is.na(condition_start_date)) %>%
  mutate(days_between = condition_start_date - drug_exposure_start_date) %>%
  select(person_id, drug_exposure_start_date, condition_start_date, days_between) %>%
  head()

You can also compare dates with `<` and `>`. An earlier date counts as smaller.

In [ ]:
as.Date("2010-01-01") < as.Date("2015-01-01")

## 3. Why the order of events matters

A drug cannot have contributed to a disease if the person started taking it **after** the disease was diagnosed. So "took the drug" is not enough. For this study, a person counts as **exposed** only if they started the drug before their diagnosis, or took it and were never diagnosed.

| Situation | Status |
|---|---|
| No prescription record | Unexposed |
| Took the drug, never diagnosed | Exposed |
| Took the drug **before** diagnosis | Exposed |
| Started the drug **on or after** the diagnosis date | Unexposed (Post-Diagnosis) |

You will store this in a column called `true_exposure_status`: "true" because it takes the timing into account, and not just whether a prescription exists.

## 4. Start with a small test

Rules like these are easy to get subtly wrong. So before using them on the real table, try them on a few made-up patients for whom the right answer is obvious.

In [ ]:
timing_test <- tibble(
  person_id                = 1:6,
  drug_exposure_start_date = as.Date(c(NA, "2010-01-01", "2010-01-01", "2018-01-01", NA, "2015-01-01")),
  condition_start_date     = as.Date(c(NA, NA, "2015-01-01", "2015-01-01", "2015-01-01", "2015-01-01"))
)
timing_test

**Before running any more code, decide the status of each of the six patients using the table of rules above.** Write your answers down. (`tibble()` is the tidyverse's version of `data.frame()`.)

### A complication: comparing with a missing date

What does R say when you compare a date with a missing date?

In [ ]:
as.Date("2010-01-01") < NA

`NA`: R cannot know. This is why the rules have to deal with the missing dates **first**, before comparing any dates.

### Writing the rules with `case_when()`

`case_when()` checks its rules from top to bottom and uses the first one that is true.

In [ ]:
timing_test %>%
  mutate(
    true_exposure_status = case_when(
      # Rule 1: no prescription record
      is.na(drug_exposure_start_date) ~ "Unexposed",
      # Rule 2: took the drug, never diagnosed
      !is.na(drug_exposure_start_date) & is.na(condition_start_date) ~ "Exposed",
      # Rule 3: took the drug BEFORE the diagnosis
      drug_exposure_start_date < condition_start_date ~ "Exposed",
      # Rule 4: started the drug ON or AFTER the diagnosis date
      drug_exposure_start_date >= condition_start_date ~ "Unexposed (Post-Diagnosis)"
    )
  )

Compare with your own answers. Patients 2 and 3 are exposed. Patient 4 started the drug three years after diagnosis, and patient 6 on the day of diagnosis, so neither is. Patients 1 and 5 never took it.

In notebook A1 you saw that leaving out `.default` can silently create missing values. Here it is left out on purpose: the four rules are meant to cover every possible case. If anyone ended up with `NA`, it would mean a case had been overlooked, and you would want to see that.

## 5. Apply it to the real table

The rules work on the test cases, so now use exactly the same code on `combined_df`, and store the result.

In [ ]:
combined_df <- combined_df %>%
  mutate(
    true_exposure_status = case_when(
      is.na(drug_exposure_start_date) ~ "Unexposed",
      !is.na(drug_exposure_start_date) & is.na(condition_start_date) ~ "Exposed",
      drug_exposure_start_date < condition_start_date ~ "Exposed",
      drug_exposure_start_date >= condition_start_date ~ "Unexposed (Post-Diagnosis)"
    )
  )

combined_df %>% count(true_exposure_status)

35 exposed, 65 unexposed, and no missing values.

Notice that nobody in this dataset falls into the "Post-Diagnosis" group. That means that **if Rule 4 had been written wrongly, this count would look exactly the same**, and you would never know. The small test table is what showed that the rule works. Real data will not always contain the cases that reveal a mistake, so make your own.

## 6. Age

**Your turn 1.** Add a column `age` to `combined_df`, calculated as 2026 minus `year_of_birth`. Store the result back in `combined_df`, then find the mean age.

*Expected result: 52.91*

In [ ]:
# Your code here

## 7. Yes/no variables as 1 and 0

For statistical models, a yes/no variable is best stored as the numbers 1 (yes) and 0 (no).

A person has the condition if they have a diagnosis record, that is, if `condition_concept_id` is not missing.

In [ ]:
combined_df <- combined_df %>%
  mutate(condition_status = if_else(is.na(condition_concept_id), 0, 1))

combined_df %>% count(condition_status)

30 people with the condition, 70 without. Because the values are 1 and 0, `sum()` counts the people with the condition and `mean()` gives the proportion.

In [ ]:
sum(combined_df$condition_status)
mean(combined_df$condition_status)

**Your turn 2.** Add a column `exposure_status` that is 1 if `true_exposure_status` is "Exposed" and 0 otherwise. Store it back in `combined_df`, then count the two groups.

*Expected result: 65 people with 0, 35 people with 1*

In [ ]:
# Your code here

## 8. A first look at the research question

`count()` with two columns gives every combination: a simple cross-tabulation of exposure against outcome.

In [ ]:
combined_df %>% count(exposure_status, condition_status)

(If this gives an error about `exposure_status`, finish the exercise above first.)

Among the 65 unexposed people, 9 have the condition. Among the 35 exposed people, 21 do. That looks like a large difference. Whether it is statistically convincing is a question for the notebooks on statistics.

## 9. Save your work

Everything you have built exists only in R's memory, and Colab's memory is wiped when the session ends. `write_csv()` saves a dataframe as a CSV file. You give it the dataframe and a file name. (The saved table is complete only if you have done both exercises above.)

In [ ]:
write_csv(combined_df, "cleaned_dataset.csv")

In Colab, the file appears in the **Files** panel (the folder icon in the left sidebar). It is deleted when the session ends, so download it from that panel if you want to keep it. In the *All of Us* Researcher Workbench, you save files to your workspace instead.

Good file names have no spaces and say what is inside. If you include a date, write it as year-month-day, for example `cleaned_dataset_2026-05-21.csv`.

---
## Recap

| Code | What it does |
|---|---|
| `date2 - date1` | Number of days between two dates |
| `date1 < date2` | Is date1 earlier? |
| `case_when(rule1, rule2, ...)` | Rules are checked top to bottom; deal with missing values first |
| `tibble(a = ..., b = ...)` | Make a small table by hand, for testing |
| `if_else(is.na(x), 0, 1)` | 0 if missing, 1 if present |
| `count(a, b)` | Count every combination of two columns |
| `write_csv(df, "file.csv")` | Save a dataframe to a file |

---
## Solutions

Try each exercise before looking. There is often more than one correct way to write it.

**Your turn 1**

```r
combined_df <- combined_df %>% mutate(age = 2026 - year_of_birth)
mean(combined_df$age)
```

**Your turn 2**

```r
combined_df <- combined_df %>%
  mutate(exposure_status = if_else(true_exposure_status == "Exposed", 1, 0))

combined_df %>% count(exposure_status)
```

---
## Next

Continue with [A4: Exploring and Plotting](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/A4_Exploring_and_Plotting.ipynb).